# RoSBERTa: base vs fine-tuned (retrieval бенчмарк)

Сравнение `ai-forever/ru-en-RoSBERTa` (zero-shot) и дообученного варианта
(`models/final/bi-encoder`, Augmented SBERT — CoSENTLoss на gold_train + silver).


In [ ]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали из подпапки
import os
from pathlib import Path
_p = Path.cwd()
while _p.name != 'thesis' and _p.parent != _p:
    _p = _p.parent
if _p.name == 'thesis':
    os.chdir(_p)
print('CWD:', Path.cwd())


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import html
import time
import gc
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
from scipy import stats
from IPython.display import HTML, display

# sentence-transformers 5.x ↔ transformers 4.57 совместимость
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import lancedb

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')


Файл считает три блока:

1. **Метрики на golden_eval.parquet** (1598 пар) — корреляция предсказанных
   косинусов с эталонными скорами (Spearman / Pearson / Kendall) и retrieval
   внутри 1598 кандидатов (Hit@K, NDCG@K, MRR).
2. **Метрики LanceDB-retrieval** на 19 ручных GT-парах — поиск в реальном
   индексе из 50 000 постов, ранг целевого поста в top-K.
3. **Визуальная таблица** top-5 по обеим моделям для каждой GT-пары.

Существующий `rosberta-base-vs-fine-tuned.ipynb` остался — он считает только
корреляции на parquet и не использует LanceDB. Этот файл — расширенная
retrieval-версия.


In [ ]:
# ==================== МОДЕЛИ ДЛЯ СРАВНЕНИЯ ====================
# Каждая запись = одна модель в этом бенчмарке. Чтобы выключить — закомментируй.

MODELS = [
    {
        'key':          'rosberta_base',
        'display_name': 'RoSBERTa base',
        'model_path':   'ai-forever/ru-en-RoSBERTa',
        'table_name':   'rosberta-base-50k',
        'doc_prefix':   '',
        'query_prefix': '',
        'color':        '#cfe2ff',
    },
    {
        'key':          'rosberta_ft',
        'display_name': 'RoSBERTa fine-tuned',
        'model_path':   'models/final/bi-encoder',
        'table_name':   'rosberta-fine-tuned-50k',
        'doc_prefix':   '',
        'query_prefix': '',
        'color':        '#9ec5fe',
    },
]

# Закомментированный слот для bge-m3 fine-tuned — раскомментируй,
# когда модель и таблица 'bge-m3-fine-tuned-50k' будут готовы:
# MODELS.append({
#     'key':          'bge_ft',
#     'display_name': 'USER-bge-m3 fine-tuned',
#     'model_path':   'models/bi-encoder-bge-m3-finetuned',
#     'table_name':   'bge-m3-fine-tuned-50k',
#     'doc_prefix':   '',
#     'query_prefix': '',
#     'color':        '#8ce99a',
# })

# --------- общие параметры ---------
LANCEDB_PATH    = './lancedb_store'
EVAL_PARQUET    = 'data/golden/golden_eval.parquet'
GT_POSTS_JSON   = 'ground_truth_posts.json'
GT_PAIRS_JSON   = 'ground_truth_pairs.json'
TOP_K_VISUAL    = 5     # сколько постов показать в визуальной таблице
TOP_K_RETRIEVAL = 20    # глубина для Hit@K и MRR в LanceDB-блоке
BATCH_SIZE      = 64    # для encode на golden_eval (1598 строк)

print(f'Будет сравниваться моделей: {len(MODELS)}')
for m in MODELS:
    print(f"  - {m['display_name']:30s} ({m['table_name']})")


In [ ]:
# Проверяем, что все нужные LanceDB-таблицы существуют ДО загрузки моделей
db = lancedb.connect(LANCEDB_PATH)
available = set(db.table_names())
missing = [m for m in MODELS if m['table_name'] not in available]
if missing:
    msg = '\n'.join(f"  - {m['display_name']}: нет таблицы {m['table_name']}" for m in missing)
    raise RuntimeError(
        f'В {LANCEDB_PATH} отсутствуют таблицы для следующих моделей:\n{msg}\n\n'
        f'Сначала прогони thesis/db/create-all-dbs.ipynb для нужных моделей, '
        f'либо закомментируй их в MODELS выше.'
    )
print(f'Все {len(MODELS)} таблиц на месте.')

# Заодно проверяем golden_eval.parquet и GT-файлы
for path in [EVAL_PARQUET, GT_POSTS_JSON, GT_PAIRS_JSON]:
    assert os.path.exists(path), f'Нет файла: {path}'
print('golden_eval.parquet, ground_truth_*.json — на месте.')


In [ ]:
# Загружаем golden_eval.parquet и ground_truth_pairs.json
from datasets import load_dataset

eval_ds = load_dataset('parquet', data_files=EVAL_PARQUET, split='train')
descriptions = list(eval_ds['product_desc'])
posts        = list(eval_ds['post_text'])
gold_scores  = np.array(eval_ds['score'], dtype=float)
print(f'golden_eval: {len(descriptions):,} пар')
print(f'  пример desc:  {descriptions[0][:80]}...')
print(f'  пример post:  {posts[0][:80]}...')
print(f'  пример score: {gold_scores[0]:.3f}')

with open(GT_POSTS_JSON, encoding='utf-8') as f:
    gt_posts = json.load(f)
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    gt_pairs = json.load(f)
post_text_by_num = {p['post_id']: p['text'] for p in gt_posts}
print(f'GT: {len(gt_pairs)} пар, {len(gt_posts)} эталонных постов')


## A. Метрики на golden_eval.parquet (1598 пар)

Кодируем все 1598 описаний и постов каждой моделью, считаем cosine на парах
(для корреляций с эталонными скорами) и cosine-матрицу (для retrieval-метрик
внутри 1598 кандидатов: каждый desc_i ищет post_i среди 1597 дистракторов).


In [ ]:
# Функция: посчитать все 'статистические' метрики одной модели на golden_eval
def compute_parquet_metrics(model, model_cfg, descriptions, posts, gold_scores):
    """Возвращает dict с метриками: spearman/pearson/kendall + NDCG/MRR/Hit@K."""
    desc_in = [model_cfg['query_prefix'] + d for d in descriptions] if model_cfg['query_prefix'] else descriptions
    post_in = [model_cfg['doc_prefix']   + p for p in posts]        if model_cfg['doc_prefix']   else posts

    desc_embs = model.encode(desc_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)
    post_embs = model.encode(post_in, batch_size=BATCH_SIZE, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False)

    # Cosine на ПАРАХ (i, i) — это и есть предсказанный скор для (desc_i, post_i)
    pair_scores = (desc_embs * post_embs).sum(dim=1).cpu().numpy()

    # Корреляции с эталоном
    spearman_r, _ = stats.spearmanr(gold_scores, pair_scores)
    pearson_r, _  = stats.pearsonr(gold_scores, pair_scores)
    kendall_t, _  = stats.kendalltau(gold_scores, pair_scores)

    # Ranking-метрики: для каждого desc_i кандидаты — все 1598 постов;
    # истинный — post_i (релевантность = gold_scores[i]); пары с gold=0 пропускаем.
    sim_matrix = cos_sim(desc_embs, post_embs).cpu().numpy()
    n = len(descriptions)

    ndcg_at = {1: [], 3: [], 5: [], 10: []}
    rrs, hit_at = [], {1: 0, 3: 0, 5: 0, 10: 0}
    used = 0
    for i in range(n):
        if gold_scores[i] <= 0:
            continue
        used += 1
        sims = sim_matrix[i]
        order = np.argsort(-sims)
        rank = int(np.where(order == i)[0][0]) + 1
        rrs.append(1.0 / rank)
        for k in hit_at:
            if rank <= k:
                hit_at[k] += 1
        # NDCG@k: одна релевантная позиция с gain = gold_scores[i]
        for k in ndcg_at:
            if rank <= k:
                dcg = gold_scores[i] / np.log2(rank + 1)
                idcg = gold_scores[i] / np.log2(2)  # idealный — на позиции 1
                ndcg_at[k].append(dcg / idcg)
            else:
                ndcg_at[k].append(0.0)

    return {
        'spearman':   spearman_r,
        'pearson':    pearson_r,
        'kendall':    kendall_t,
        'mrr':        float(np.mean(rrs)) if rrs else 0.0,
        'hit@1':      hit_at[1] / used if used else 0.0,
        'hit@3':      hit_at[3] / used if used else 0.0,
        'hit@5':      hit_at[5] / used if used else 0.0,
        'hit@10':     hit_at[10] / used if used else 0.0,
        'ndcg@1':     float(np.mean(ndcg_at[1])),
        'ndcg@3':     float(np.mean(ndcg_at[3])),
        'ndcg@5':     float(np.mean(ndcg_at[5])),
        'ndcg@10':    float(np.mean(ndcg_at[10])),
        'eval_pairs': used,
    }


## B. Метрики LanceDB-retrieval на 19 GT парах

Реалистичный сценарий: для каждого описания товара модель ищет ответ в
**настоящем индексе из 50 000 постов**. Считаем, найдётся ли целевой пост
в top-K и на каком ранге.


In [ ]:
# Функция: метрики LanceDB-retrieval на 19 GT парах для одной модели
def compute_lancedb_metrics(model, model_cfg, gt_pairs, post_text_by_num, top_k=20):
    table = db.open_table(model_cfg['table_name'])
    qprefix = model_cfg['query_prefix']

    ranks = []      # ранг целевого поста среди top_k (None — не нашёлся)
    rrs   = []
    hit   = {5: 0, 10: 0, 20: 0}
    found_count = 0

    for pair in gt_pairs:
        target_text = post_text_by_num[pair['post_num']].strip()
        q_in = (qprefix + pair['description']) if qprefix else pair['description']
        qvec = model.encode([q_in], normalize_embeddings=True)[0].tolist()
        rows = (table.search(qvec, query_type='vector')
                     .limit(top_k)
                     .select(['text', 'channel', 'category'])
                     .to_list())
        rank = None
        for i, r in enumerate(rows, 1):
            if r['text'].strip() == target_text:
                rank = i
                break
        ranks.append(rank)
        if rank is not None:
            found_count += 1
            rrs.append(1.0 / rank)
            for k in hit:
                if rank <= k:
                    hit[k] += 1
        else:
            rrs.append(0.0)

    n = len(gt_pairs)
    return {
        'ranks':     ranks,
        'mrr':       float(np.mean(rrs)),
        'hit@5':     hit[5] / n,
        'hit@10':    hit[10] / n,
        'hit@20':    hit[20] / n,
        'found':     f'{found_count}/{n}',
        'mean_rank': float(np.mean([r for r in ranks if r is not None])) if found_count else float('nan'),
    }


In [ ]:
# ============================================================
# ГЛАВНЫЙ ЦИКЛ ПО МОДЕЛЯМ: загружаем, считаем оба блока метрик
# ============================================================
results = OrderedDict()      # key -> {parquet_metrics, lancedb_metrics}
loaded_models = OrderedDict() # key -> SentenceTransformer (нужен ниже для визуала)

for cfg in MODELS:
    print(f"\n--- {cfg['display_name']} ({cfg['model_path']}) ---")
    t0 = time.time()
    model = SentenceTransformer(cfg['model_path'], device=DEVICE)
    model.max_seq_length = 256
    print(f'  загружена за {time.time()-t0:.0f}с, dim={model.get_sentence_embedding_dimension()}')

    print('  parquet metrics...', end=' ', flush=True)
    t0 = time.time()
    pm = compute_parquet_metrics(model, cfg, descriptions, posts, gold_scores)
    print(f'{time.time()-t0:.0f}с')
    print(f'    Spearman={pm["spearman"]:.4f}  MRR={pm["mrr"]:.4f}  Hit@5={pm["hit@5"]:.4f}')

    print('  lancedb metrics...', end=' ', flush=True)
    t0 = time.time()
    lm = compute_lancedb_metrics(model, cfg, gt_pairs, post_text_by_num,
                                  top_k=TOP_K_RETRIEVAL)
    print(f'{time.time()-t0:.0f}с')
    print(f'    found={lm["found"]}  MRR={lm["mrr"]:.4f}  Hit@5={lm["hit@5"]:.4f}')

    results[cfg['key']] = {'cfg': cfg, 'parquet': pm, 'lancedb': lm}
    loaded_models[cfg['key']] = model

print('\nГотово.')


## Сводки


In [ ]:
# Сводная таблица: метрики на golden_eval.parquet
rows = []
for key, r in results.items():
    p = r['parquet']
    rows.append({
        'Модель':     r['cfg']['display_name'],
        'Spearman ρ': round(p['spearman'], 4),
        'Pearson r':  round(p['pearson'],  4),
        'Kendall τ':  round(p['kendall'],  4),
        'MRR':        round(p['mrr'],      4),
        'Hit@1':      round(p['hit@1'],    4),
        'Hit@5':      round(p['hit@5'],    4),
        'Hit@10':     round(p['hit@10'],   4),
        'NDCG@5':     round(p['ndcg@5'],   4),
        'NDCG@10':    round(p['ndcg@10'],  4),
    })
df_parquet = pd.DataFrame(rows)
print(f'Метрики на golden_eval.parquet ({results[next(iter(results))]["parquet"]["eval_pairs"]} пар с gold>0)')
display(df_parquet.style.background_gradient(cmap='YlGn',
        subset=['Spearman ρ','Pearson r','Kendall τ','MRR','Hit@1','Hit@5','Hit@10','NDCG@5','NDCG@10']))


In [ ]:
# Сводная таблица: LanceDB-retrieval на 19 GT парах
rows = []
for key, r in results.items():
    l = r['lancedb']
    rows.append({
        'Модель':       r['cfg']['display_name'],
        'Found':        l['found'],
        f'Hit@5':       round(l['hit@5'],  4),
        f'Hit@10':      round(l['hit@10'], 4),
        f'Hit@20':      round(l['hit@20'], 4),
        'MRR':          round(l['mrr'],    4),
        'Mean rank':    round(l['mean_rank'], 1) if not np.isnan(l['mean_rank']) else '—',
    })
df_lancedb = pd.DataFrame(rows)
print(f'Метрики LanceDB-retrieval (поиск в индексе из 50k постов, {len(gt_pairs)} запросов)')
display(df_lancedb.style.background_gradient(cmap='YlGn',
        subset=['Hit@5','Hit@10','Hit@20','MRR']))


In [ ]:
# Сводка: ранг целевого поста по каждой GT-паре, по каждой модели
def fmt_rank(r):
    if r is None:
        bg, bd = '#f8d7da', '#f1aeb5'; txt = '—'
    elif r <= 5:
        bg, bd = '#d4edda', '#28a745'; txt = f'#{r}'
    elif r <= 20:
        bg, bd = '#fff3cd', '#ffc107'; txt = f'#{r}'
    else:
        bg, bd = '#f8d7da', '#f1aeb5'; txt = f'#{r}'
    return (f'<span style="background:{bg};color:#000;padding:2px 8px;'
            f'border-radius:3px;font-weight:bold;border:1px solid {bd};">{txt}</span>')

def esc(s):
    return html.escape(str(s)).replace('\n', '<br>')

header_cells = ''.join(
    f'<th style="padding:8px 10px;color:#000;border-bottom:2px solid #adb5bd;background:{r["cfg"]["color"]};">'
    f'{esc(r["cfg"]["display_name"])}</th>'
    for r in results.values()
)

body_rows = []
for idx, pair in enumerate(gt_pairs, 1):
    cells = ''.join(
        f'<td style="padding:6px 10px;text-align:center;border-bottom:1px solid #e9ecef;">'
        f'{fmt_rank(r["lancedb"]["ranks"][idx-1])}</td>'
        for r in results.values()
    )
    body_rows.append(
        f'<tr style="background:#ffffff;color:#000;">'
        f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{idx}</td>'
        f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">GT #{pair["post_num"]}</td>'
        f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{esc(pair.get("imt_name",""))[:80]}</td>'
        f'{cells}</tr>'
    )
rows_html = ''.join(body_rows)

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;margin:16px 0;color:#000;">
    <div style="font-size:15px;margin-bottom:8px;color:#000;"><b>Ранг целевого поста по каждой GT-паре</b>
    (зелёный — top-5, жёлтый — top-20, красный — за пределами top-{TOP_K_RETRIEVAL})</div>
    <table style="border-collapse:collapse;font-size:13px;background:#ffffff;color:#000;">
        <thead><tr style="background:#e9ecef;color:#000;">
            <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">#</th>
            <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">GT пост</th>
            <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">Товар</th>
            {header_cells}
        </tr></thead>
        <tbody>{rows_html}</tbody>
    </table>
</div>
'''))


## C. Визуал: top-5 по каждой модели для каждой GT-пары

Для каждой из 19 пар — описание товара, целевой пост и таблица top-5 по
каждой модели бок-о-бок. Если целевой пост попал в top-5 у модели — её
ячейка подсвечивается зелёным.


In [ ]:
# Функция: для одной GT-пары показать запрос и таблицу top-K по каждой модели
def render_pair_topk(idx, total, pair, target_text, model_topk, top_k=5):
    """model_topk: OrderedDict[key -> list of {text, channel, category} dicts]"""
    target_norm = target_text.strip()

    # Заголовки колонок
    col_headers = ''.join(
        f'<th style="padding:10px;text-align:left;background:{cfg["color"]};'
        f'border-bottom:2px solid #495057;color:#000;font-weight:bold;'
        f'border-right:1px solid #adb5bd;width:{round(100/len(model_topk), 2)}%;">'
        f'{esc(cfg["display_name"])}</th>'
        for cfg in (results[k]['cfg'] for k in model_topk.keys())
    )

    # Тело таблицы: строка на каждый ранг 1..top_k
    body = ''
    for rank in range(1, top_k + 1):
        cells = ''
        for key, rows in model_topk.items():
            if rank > len(rows):
                cells += '<td style="padding:10px;color:#000;border-right:1px solid #e9ecef;border-bottom:1px solid #e9ecef;vertical-align:top;">—</td>'
                continue
            r = rows[rank - 1]
            is_target = (r['text'].strip() == target_norm)
            bg = '#d4edda' if is_target else '#ffffff'
            border = '4px solid #28a745' if is_target else 'none'
            star = ' ★' if is_target else ''
            text_full = r['text'].replace('\n', ' ')
            cells += (
                f'<td style="padding:10px;color:#000;background:{bg};'
                f'border-left:{border};border-right:1px solid #e9ecef;'
                f'border-bottom:1px solid #e9ecef;vertical-align:top;font-size:12px;">'
                f'<div style="font-weight:bold;font-size:11px;margin-bottom:4px;color:#495057;">'
                f'#{rank}{star} · @{esc(r["channel"])} · {esc(r.get("category",""))}</div>'
                f'<div style="line-height:1.4;color:#000;">{esc(text_full)}</div>'
                f'</td>'
            )
        body += f'<tr>{cells}</tr>'

    return HTML(f'''
    <div style="border:2px solid #495057;border-radius:8px;margin:32px 0;
                background:#ffffff;font-family:system-ui,sans-serif;
                overflow:hidden;color:#000;">
        <div style="background:#e9ecef;padding:14px 20px;border-bottom:1px solid #ced4da;">
            <div style="font-size:17px;font-weight:bold;color:#000;">
                Пара {idx}/{total} · GT post #{pair["post_num"]}
            </div>
            <div style="font-size:13px;color:#495057;margin-top:4px;">
                {esc(pair.get("imt_name",""))} · {esc(pair.get("subj_name",""))}
            </div>
        </div>
        <div style="padding:14px 20px;background:#e7f3ff;border-bottom:1px solid #cfe2ff;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Запрос (описание товара)</div>
            <div style="color:#000;font-size:13px;line-height:1.5;">{esc(pair["description"])}</div>
        </div>
        <div style="padding:14px 20px;background:#d4edda;border-bottom:1px solid #c3e6cb;">
            <div style="font-weight:bold;color:#000;font-size:12px;text-transform:uppercase;
                        margin-bottom:6px;letter-spacing:0.5px;">Целевой пост (должен попасть в top-{top_k})</div>
            <div style="color:#000;font-size:13px;line-height:1.5;white-space:pre-wrap;">{esc(target_text)}</div>
        </div>
        <table style="width:100%;border-collapse:collapse;background:#ffffff;table-layout:fixed;">
            <thead><tr>{col_headers}</tr></thead>
            <tbody>{body}</tbody>
        </table>
    </div>
    ''')


In [ ]:
# ============================================================
# ВИЗУАЛ: для каждой из 19 GT пар — топ-5 по каждой модели
# ============================================================
for idx, pair in enumerate(gt_pairs, 1):
    target_text = post_text_by_num[pair['post_num']]
    model_topk = OrderedDict()
    for cfg in MODELS:
        model = loaded_models[cfg['key']]
        table = db.open_table(cfg['table_name'])
        q_in = (cfg['query_prefix'] + pair['description']) if cfg['query_prefix'] else pair['description']
        qvec = model.encode([q_in], normalize_embeddings=True)[0].tolist()
        rows = (table.search(qvec, query_type='vector')
                     .limit(TOP_K_VISUAL)
                     .select(['text', 'channel', 'category'])
                     .to_list())
        model_topk[cfg['key']] = rows
    display(render_pair_topk(idx, len(gt_pairs), pair, target_text, model_topk, top_k=TOP_K_VISUAL))


In [ ]:
# Освобождаем VRAM
for k in list(loaded_models.keys()):
    del loaded_models[k]
loaded_models.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Память освобождена.')
